# LLM OpenAI Tutorial

By: Konthee Boonemeeprakob

Super Ai Engineer -- LLM Tutorial

=============================================

Notebook นี้เป็นบทสอนการใช้งาน LLM แบบค่อยเป็นค่อยไป โดยใช้ `openai` Python library เป็นแกนหลัก และรันบน Google Colab เป็นหลัก

หัวข้อที่ครอบคลุม:
- การตั้งค่าและเชื่อมต่อ OpenAI API
- การใช้ `messages` และ history สำหรับ multi-turn conversation
- การบังคับผลลัพธ์ให้เป็น JSON
- การใช้ structured output ด้วย `pydantic`
- การทำ tool calling เพื่อค้นข้อมูลแบบ deep research ด้วย DuckDuckGo
- การใช้ Gemini ผ่าน OpenAI-compatible endpoint สำหรับ OCR และ vision


## 1) Setup

Notebook นี้ใช้ package หลัก 3 กลุ่ม:
- `openai` สำหรับเรียก LLM
- `pydantic` สำหรับ structured output
- `duckduckgo-search` สำหรับตัวอย่าง web search tool

ใน Google Colab ให้ไปที่ `Secrets` แล้วเพิ่มค่าเหล่านี้:
- `GROQ_API`    https://console.groq.com/keys
- `GEMINI_API_KEY` https://aistudio.google.com/u/1/api-keys

หมายเหตุ: ส่วน DuckDuckGo ไม่ต้องใช้ API key แต่คุณภาพและความเสถียรอาจไม่เท่า commercial search APIs


In [ ]:
!pip -q install openai pydantic duckduckgo-search ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 51.0 MB/s eta 0:00:00


In [ ]:
import base64
import json
from typing import Any

from duckduckgo_search import DDGS
from google.colab import userdata
from openai import OpenAI
from pydantic import BaseModel, Field
from IPython.display import Markdown, display

from datetime import datetime, UTC
from zoneinfo import ZoneInfo
import warnings

warnings.filterwarnings(
    "ignore",
    message=r".*datetime\.datetime\.utcnow\(\) is deprecated.*",
    category=DeprecationWarning,
)

dt_utc = datetime.now(UTC)
dt_th = dt_utc.astimezone(ZoneInfo("Asia/Bangkok"))
print(dt_th)


2026-03-24 11:52:44.621784+07:00


In [ ]:
OPENAI_MODEL = "openai/gpt-oss-120b"

def build_openai_client() -> OpenAI:
    return OpenAI(
        base_url = "https://api.groq.com/openai/v1",
        api_key=userdata.get('GROQ_API')
        )



def pretty_print_json(data: Any) -> None:
    print(json.dumps(data, indent=2, ensure_ascii=False))


def duckduckgo_search(query: str, max_results: int = 5) -> dict:
    results = []
    with DDGS() as ddgs:
        for item in ddgs.text(query, max_results=max_results):
            results.append(
                {
                    "title": item.get("title"),
                    "href": item.get("href"),
                    "body": item.get("body"),
                }
            )
    return {"query": query, "results": results}


client = build_openai_client()



## 2) Basic Chat Completion

`messages` คือ list ของบทสนทนา แต่ละรายการจะมี `role` เช่น:
- `system` ใช้กำหนดพฤติกรรมหรือบทบาทของโมเดล
- `user` คือคำถามจากผู้ใช้
- `assistant` คือคำตอบจากโมเดล

พารามิเตอร์ที่พบบ่อย:
- `temperature` ยิ่งสูงยิ่งหลากหลาย
- `max_tokens` จำกัดความยาวของคำตอบ


In [ ]:
## without thinking
messages = [
    {"role": "system", "content": "คุณเป็นผู้ช่วยสอน AI ที่อธิบายสั้น ชัด และเป็นภาษาไทย"},
    {"role": "user", "content": "อธิบายว่า LLM คืออะไร แบบสั้น 3 บรรทัด"},
]

response = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=messages,
    temperature=0.3,
    max_tokens=250,
    # reasoning_effort="low",
)

thinking = response.choices[0].message.reasoning
print(f"Thinking :\n {thinking}")

basic_answer = response.choices[0].message.content
print(f"Answer:\n {basic_answer}" )

Thinking :
 The user wants a short explanation of LLM in Thai, 3 lines. Provide concise.
Answer:
 LLM (Large Language Model) คือโมเดลปัญญาประดิษฐ์ที่ฝึกด้วยข้อมูลข้อความจำนวนมหาศาลเพื่อทำความเข้าใจและสร้างภาษาได้เหมือนมนุษย์  
มันใช้สถาปัตยกรรมเช่น Transformer เพื่อจับความสัมพันธ์ระหว่างคำในประโยคและบริบทกว้าง ๆ  
ผลลัพธ์คือสามารถตอบคำถาม, สรุปข้อมูล, เขียนบทความ หรือทำงานภาษาอื่น ๆ ได้อย่างอัตโนมัติ.


In [ ]:
# High reasoning

messages = [
    {"role": "system", "content": "คุณเป็นผู้ช่วยสอน AI ที่อธิบายสั้น ชัด และเป็นภาษาไทย"},
    {"role": "user", "content": "อธิบายว่า LLM คืออะไร แบบสั้น 3 บรรทัด"},
]

response = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=messages,
    temperature=0.3,
    max_tokens=1250,
    reasoning_effort="high",
)

thinking = response.choices[0].message.reasoning
print(f"Thinking :\n {thinking}")

basic_answer = response.choices[0].message.content
print(f"Answer:\n {basic_answer}" )

Thinking :
 The user asks: "อธิบายว่า LLM คืออะไร แบบสั้น 3 บรรทัด". They want a short explanation of LLM (Large Language Model) in Thai, in 3 lines. The developer instruction says: "คุณเป็นผู้ช่วยสอน AI ที่อธิบายสั้น ชัด และเป็นภาษาไทย". So we need to comply: short, clear, Thai. Provide 3 lines. So we need to produce a short explanation in Thai, exactly 3 lines. Possibly each line separated by newline. Should be concise. Something like:

- LLM (Large Language Model) คือโมเดลปัญญาประดิษฐ์ที่เรียนรู้จากข้อความจำนวนมหาศาล.
- มันใช้สถาปัตยกรรม neural network เช่น Transformer เพื่อทำนายคำถัดไป.
- สามารถทำงานหลายอย่างเช่น แปลภาษา สรุปข้อความ และตอบคำถาม.

That's 3 lines. That satisfies. Ensure it's short, clear, Thai. Provide exactly 3 lines. Should not add extra lines. So answer accordingly.
Answer:
 LLM (Large Language Model) คือโมเดลปัญญาประดิษฐ์ที่เรียนรู้จากข้อความจำนวนมหาศาล  
ใช้สถาปัตยกรรม Neural Network เช่น Transformer เพื่อทำนายคำถัดไป  
สามารถทำงานหลายอย่าง เช่น แปลภาษา สรุปข้อค

## 3) History: สนทนาต่อเนื่องหลายรอบ

จุดสำคัญของ history คือเราต้องส่งบทสนทนาก่อนหน้าเข้าไปด้วย เพื่อให้โมเดลเห็นบริบทและตอบต่อเนื่องได้อย่างถูกต้อง


In [ ]:
conversation_history = [
    {"role": "system", "content": "คุณเป็นผู้ช่วยแนะนำการเรียน LLM เป็นภาษาไทย"},
    {"role": "user", "content": "ฉันเพิ่งเริ่มเรียน LLM ควรเริ่มจากอะไร"},
]

first_turn = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=conversation_history,
    temperature=0.4,
)

first_answer = first_turn.choices[0].message.content
conversation_history.append({"role": "assistant", "content": first_answer})
print("คำตอบรอบแรก:\n")
print(first_answer)

คำตอบรอบแรก:

### เริ่มต้นเรียนรู้ Large Language Model (LLM) อย่างเป็นระบบ

> **เคล็ดลับสำคัญ**  
> 1️⃣ **พื้นฐานคณิตศาสตร์ & โปรแกรมมิ่ง** – ไม่จำเป็นต้องเป็นผู้เชี่ยวชาญระดับสูง แต่ต้องเข้าใจพื้นฐานของเชิงเส้น, แคลคูลัส, ความน่าจะเป็น และการเขียนโค้ด Python  
> 2️⃣ **ทำตามโครงการจริง** – ทุกครั้งที่เรียนทฤษฎี ควรทำแบบฝึกหัดหรือโครงการเล็ก ๆ เพื่อให้ความรู้ “จับต้องได้”  
> 3️⃣ **เรียนแบบเป็นชั้น** – เริ่มจากพื้นฐานของ Machine Learning → Deep Learning → NLP → Transformers → LLM  

---

## 1️⃣ ทำความเข้าใจพื้นฐาน Machine Learning (ML)

| หัวข้อ | ทำอะไร | แหล่งเรียนรู้ (ภาษาไทย/อังกฤษ) |
|--------|--------|--------------------------------|
| **แนวคิดพื้นฐาน** (Supervised, Unsupervised, Reinforcement) | เข้าใจประเภทของปัญหาและวิธีการประเมินผล | 📚 *คอร์ส Coursera: Machine Learning (Andrew Ng)* – มีซับไตเติลไทย |
| **คณิตศาสตร์พื้นฐาน** (Linear Algebra, Calculus, Probability) | เข้าใจเวกเตอร์, เมทริกซ์, การทำ Gradient Descent | 📖 *หนังสือ “คณิตศาสตร์สำหรับ Data Science”* (ภาษาไทย) |
| **

In [ ]:
conversation_history.append(
    {"role": "user", "content": "ถ้าฉันมีเวลาแค่วันละ 30 นาที ช่วยจัดแผน 7 วันให้หน่อย"}
)

second_turn = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=conversation_history,
    temperature=0.4,
)

second_answer = second_turn.choices[0].message.content
conversation_history.append({"role": "assistant", "content": second_answer})
print("คำตอบรอบที่สอง:\n")
print(second_answer)

คำตอบรอบที่สอง:

## แผนเรียน “LLM เบื้องต้น” 7 วัน – 30 นาทีต่อวัน  
> **หลักการ** – ใช้เวลา 30 นาทีต่อวันให้ได้ผลสูงสุด:  
> 1️⃣ **15 นาที** ดู/อ่านเนื้อหา (วิดีโอสั้น, บทความสรุป)  
> 2️⃣ **10 นาที** ทำแบบฝึกหัดหรือทดลองโค้ดสั้น ๆ  
> 3️⃣ **5 นาที** สรุปสิ่งที่เรียน (บันทึกในโน้ตหรือ Google Docs)  

---

### 📅 ตาราง 7 วัน

| วัน | หัวข้อ (30 นาที) | 15 นาที – เรียน | 10 นาที – ทำ | 5 นาที – สรุป/บันทึก |
|-----|------------------|------------------|--------------|----------------------|
| **Day 1** | **ทำความรู้จัก LLM & Python เบื้องต้น** | • วิดีโอ 5 min: “What is a Large Language Model?” (OpenAI) <br>• อ่านบทสรุป 5 min: <https://huggingface.co/blog/large-language-models> <br>• ดู 5 min: “Python for AI – 5‑minute intro” (YouTube) | เปิด **Google Colab** แล้วรันโค้ด “Hello World” ของ Python (`print("Hello LLM!")`) | เขียนบันทึก 2‑3 บรรทัดว่า LLM คืออะไร, ทำไมต้องใช้ Python |
| **Day 2** | **พื้นฐานเชิงเส้น (Linear Algebra) – เวกเตอร์** | • บทความ 5 min: “เวกเตอร์คืออะไร? (Thai)” <br

In [ ]:
display(Markdown(second_answer))

## แผนเรียน “LLM เบื้องต้น” 7 วัน – 30 นาทีต่อวัน  
> **หลักการ** – ใช้เวลา 30 นาทีต่อวันให้ได้ผลสูงสุด:  
> 1️⃣ **15 นาที** ดู/อ่านเนื้อหา (วิดีโอสั้น, บทความสรุป)  
> 2️⃣ **10 นาที** ทำแบบฝึกหัดหรือทดลองโค้ดสั้น ๆ  
> 3️⃣ **5 นาที** สรุปสิ่งที่เรียน (บันทึกในโน้ตหรือ Google Docs)  

---

### 📅 ตาราง 7 วัน

| วัน | หัวข้อ (30 นาที) | 15 นาที – เรียน | 10 นาที – ทำ | 5 นาที – สรุป/บันทึก |
|-----|------------------|------------------|--------------|----------------------|
| **Day 1** | **ทำความรู้จัก LLM & Python เบื้องต้น** | • วิดีโอ 5 min: “What is a Large Language Model?” (OpenAI) <br>• อ่านบทสรุป 5 min: <https://huggingface.co/blog/large-language-models> <br>• ดู 5 min: “Python for AI – 5‑minute intro” (YouTube) | เปิด **Google Colab** แล้วรันโค้ด “Hello World” ของ Python (`print("Hello LLM!")`) | เขียนบันทึก 2‑3 บรรทัดว่า LLM คืออะไร, ทำไมต้องใช้ Python |
| **Day 2** | **พื้นฐานเชิงเส้น (Linear Algebra) – เวกเตอร์** | • บทความ 5 min: “เวกเตอร์คืออะไร? (Thai)” <br>• วิดีโอ 5 min: “Vector addition & scalar multiplication” (Khan Academy TH) <br>• อ่าน 5 min: ตัวอย่างการใช้ NumPy `np.array` | ใน Colab สร้าง 2‑3 เวกเตอร์ด้วย `np.array` แล้วทำการบวก/คูณสเกล่า | บันทึกสูตรพื้นฐาน + ตัวอย่างโค้ด |
| **Day 3** | **NumPy เบื้องต้น – การจัดการข้อมูลเป็นเมทริกซ์** | • วิดีโอ 5 min: “NumPy quick start (Thai subtitles)” <br>• อ่าน 5 min: หน้าแรกของเอกสาร NumPy <br>• ดู 5 min: ตัวอย่างการทำ `np.dot` | สร้างเมทริกซ์ 2×2 แล้วคูณกับเวกเตอร์ 2‑มิติ (`np.dot`) | สรุปคำสั่งสำคัญ: `np.array`, `np.dot`, `np.shape` |
| **Day 4** | **พื้นฐาน Machine Learning – Linear Regression** | • วิดีโอ 5 min: “Linear Regression in 5 minutes (Thai)” <br>• อ่าน 5 min: บทสรุปจาก Coursera (Supervised Learning) <br>• ดู 5 min: ตัวอย่างโค้ด Scikit‑learn `LinearRegression` | ใช้ `sklearn.datasets.load_boston` (หรือ `load_diabetes`) สร้างโมเดล Linear Regression 1‑line (`LinearRegression().fit`) | เขียนสรุปว่าโมเดลเรียนจาก “features → target” อย่างไร |
| **Day 5** | **NLP พื้นฐาน – Tokenization & Bag‑of‑Words** | • วิดีโอ 5 min: “Tokenization for Thai text (fastText)” <br>• อ่าน 5 min: บทความ “Bag‑of‑Words explained (Thai)” <br>• ดู 5 min: ตัวอย่างการใช้ `CountVectorizer` | ใช้ `sklearn.feature_extraction.text.CountVectorizer` แปลงประโยคภาษาไทย 3‑5 ประโยคเป็น BOW | บันทึกขั้นตอน Token → BOW และผลลัพธ์ที่ได้ |
| **Day 6** | **Transformers เบื้องต้น – Self‑Attention** | • วิดีโอ 5 min: “The Illustrated Transformer – Thai subtitles” <br>• อ่าน 5 min: สรุปสั้น ๆ ของ Attention (จาก blog “AI Thailand”) <br>• ดู 5 min: ตัวอย่างโค้ด `transformers` โหลดโมเดล `bert-base-thai` | ใน Colab รันโค้ดสั้น ๆ: <br>`from transformers import AutoTokenizer, AutoModel <br>tokenizer = AutoTokenizer.from_pretrained("airesearch/wangchanberta-base-att-spm-uncased") <br>model = AutoModel.from_pretrained("airesearch/wangchanberta-base-att-spm-uncased")` <br>ทำ inference กับประโยคสั้น ๆ | สรุปว่า “Attention ทำอะไร” และวิธีเรียกใช้โมเดลจาก Hugging Face |
| **Day 7** | **ทำ Mini‑Project – สร้าง Sentiment Analyzer ภาษาไทย** | • อ่าน 5 min: ตัวอย่างโค้ด Fine‑tune BERT‑Thai (จาก Hugging Face) <br>• ดู 5 min: วิดีโอ “Fine‑tune BERT on a tiny dataset (Thai)” <br>• อ่าน 5 min: วิธีประเมินผลด้วย Accuracy | ใช้ dataset เล็ก (เช่น 100 รีวิวจาก Pantip) <br>ทำขั้นตอน: <br>1. Tokenize <br>2. สร้าง `Trainer` (หรือ `pipeline`) <br>3. ทดสอบกับประโยค 2‑3 ตัว | เขียนสรุปผล Accuracy, ปัญหาที่เจอ, และแนวคิดต่อไป (เช่น เพิ่มข้อมูล, ใช้ LoRA) |

---

## 🔧 เคล็ดลับให้ 30 นาทีมีประสิทธิภาพ

| เคล็ดลับ | วิธีทำ |
|----------|--------|
| **เตรียมสภาพแวดล้อมล่วงหน้า** | สร้าง Google Colab notebook ตั้งแต่วัน 1 แล้วบันทึกลิงก์ไว้ – ไม่ต้องเปิดใหม่ทุกวัน |
| **ใช้ “Pomodoro 5‑5‑5”** | ตั้ง таймер 5 นาที (ดู/อ่าน) → 5 นาที (ทำ) → 5 นาที (สรุป) → พัก 5 นาที (รีเฟรช) |
| **บันทึกแบบ Bullet** | ใช้ Google Docs หรือ Notion บันทึกหัวข้อหลัก, คำสั่งโค้ด, คำถามที่ยังไม่เข้าใจ |
| **ตั้งคำถาม “Why?”** | หลังเรียนแต่ละหัวข้อให้ถาม “ทำไมต้องใช้วิธีนี้?” เพื่อกระตุ้นความเข้าใจลึก |
| **เชื่อมต่อกับชุมชน** | หลังจบวัน ส่งสรุปสั้น ๆ (tweet หรือโพสต์ใน Discord) – จะได้ feedback และแรงจูงใจต่อเนื่อง |

---

## 📚 แหล่งเรียนรู้สั้น ๆ (30 นาทีต่อหัวข้อ)

| ประเภท | ลิงก์ (Thai/Eng) |
|--------|-----------------|
| วิดีโอ 5 min | <https://youtu.be/6xK8cYf5c4U> (What is LLM?) |
| บทความสรุป | <https://huggingface.co/blog/large-language-models> (EN) – ใช้ Google Translate |
| Tokenizer Demo | <https://colab.research.google.com/drive/1xZyVhKc2Y3> (Thai BERT) |
| Linear Regression | <https://www.coursera.org/learn/machine-learning> – 5‑min subtitles |
| NumPy Quickstart | <https://youtu.be/8Mpc9ukltVA> (Thai subtitles) |
| Transformers Intro | <https://jalammar.github.io/illustrated-transformer/> – มีแปลไทยโดย AI Thailand |

---

## 🎯 จุดมุ่งหมายหลัง 7 วัน

1. **เข้าใจแนวคิดพื้นฐานของ LLM** (What, Why, How)  
2. **เขียนโค้ด Python เบื้องต้น** (print, ตัวแปร, NumPy)  
3. **ทำโมเดลง่าย ๆ** (Linear Regression, BOW)  
4. **เรียกใช้โมเดล Transformer ภาษาไทย** จาก Hugging Face  
5. **สร้าง Mini‑Project Sentiment Analyzer** ที่ทำงานได้ (แม้ accuracy จะต่ำ)  

หลังจากนี้คุณสามารถต่อยอดต่อไปได้โดย:
- เพิ่ม dataset ให้ใหญ่ขึ้น (Kaggle Thai Sentiment)  
- ทดลอง Fine‑tune ด้วย **LoRA** (ใช้ 30 นาทีต่อวันต่อเนื่อง)  
- เริ่มเรียนเรื่อง **RLHF** หรือ **Prompt Engineering**  

**พร้อมหรือยัง?** เริ่มวัน 1 ตอนนี้เลย – เปิด Google Colab แล้วกด “Run” เพื่อบอกว่า “Hello LLM!” 🚀

In [ ]:
pretty_print_json(conversation_history)

[
  {
    "role": "system",
    "content": "คุณเป็นผู้ช่วยแนะนำการเรียน LLM เป็นภาษาไทย"
  },
  {
    "role": "user",
    "content": "ฉันเพิ่งเริ่มเรียน LLM ควรเริ่มจากอะไร"
  },
  {
    "role": "assistant",
    "content": "### เริ่มต้นเรียนรู้ Large Language Model (LLM) อย่างเป็นระบบ\n\n> **เคล็ดลับสำคัญ**  \n> 1️⃣ **พื้นฐานคณิตศาสตร์ & โปรแกรมมิ่ง** – ไม่จำเป็นต้องเป็นผู้เชี่ยวชาญระดับสูง แต่ต้องเข้าใจพื้นฐานของเชิงเส้น, แคลคูลัส, ความน่าจะเป็น และการเขียนโค้ด Python  \n> 2️⃣ **ทำตามโครงการจริง** – ทุกครั้งที่เรียนทฤษฎี ควรทำแบบฝึกหัดหรือโครงการเล็ก ๆ เพื่อให้ความรู้ “จับต้องได้”  \n> 3️⃣ **เรียนแบบเป็นชั้น** – เริ่มจากพื้นฐานของ Machine Learning → Deep Learning → NLP → Transformers → LLM  \n\n---\n\n## 1️⃣ ทำความเข้าใจพื้นฐาน Machine Learning (ML)\n\n| หัวข้อ | ทำอะไร | แหล่งเรียนรู้ (ภาษาไทย/อังกฤษ) |\n|--------|--------|--------------------------------|\n| **แนวคิดพื้นฐาน** (Supervised, Unsupervised, Reinforcement) | เข้าใจประเภทของปัญหาและวิธีการประเมินผล | 📚 *คอร์ส Coursera: Mac

## 4) JSON Output

ถ้าเราต้องการนำผลลัพธ์ไปใช้ต่อในโปรแกรม เช่น แสดงผลใน UI หรือบันทึกลงฐานข้อมูล การให้โมเดลตอบเป็น JSON จะช่วยให้ parse และตรวจสอบข้อมูลได้ง่ายขึ้น


In [ ]:
json_messages = [
    {
        "role": "system",
        "content": (
            "ตอบกลับเป็น JSON object เท่านั้น และห้ามมีข้อความอื่นนอก JSON "
            "โดยให้มี key คือ title, difficulty, topics, next_step"
        ),
    },
    {
        "role": "user",
        "content": "ช่วยออกแบบหัวข้อ workshop เรื่อง prompt engineering สำหรับผู้เริ่มต้น 1 วัน",
    },
]

json_response = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=json_messages,
    response_format={"type": "json_object"},
    temperature=0.2,
)

json_text = json_response.choices[0].message.content
json_output = json.loads(json_text)
pretty_print_json(json_output)

{
  "title": "Prompt Engineering Workshop for Beginners (1-Day)",
  "difficulty": "Beginner",
  "topics": [
    "Introduction to AI and Large Language Models",
    "Fundamentals of Prompt Engineering",
    "Effective Prompt Design Techniques",
    "Hands‑on Prompt Crafting Exercises",
    "Iterative Prompt Refinement",
    "Evaluating Prompt Outputs",
    "Common Pitfalls and Best Practices",
    "Using Prompt Templates and Chains",
    "Ethical Considerations and Bias Mitigation",
    "Resources for Continued Learning"
  ],
  "next_step": "Apply learned techniques in personal projects, join online prompt engineering communities, explore advanced courses, and build a portfolio of effective prompts."
}


In [ ]:
print("topics =", json_output["topics"])
print("next_step =", json_output["next_step"])

topics = ['Introduction to AI and Large Language Models', 'Fundamentals of Prompt Engineering', 'Effective Prompt Design Techniques', 'Hands‑on Prompt Crafting Exercises', 'Iterative Prompt Refinement', 'Evaluating Prompt Outputs', 'Common Pitfalls and Best Practices', 'Using Prompt Templates and Chains', 'Ethical Considerations and Bias Mitigation', 'Resources for Continued Learning']
next_step = Apply learned techniques in personal projects, join online prompt engineering communities, explore advanced courses, and build a portfolio of effective prompts.


## 5) Structured Output ด้วย `pydantic`

JSON output ช่วยให้ parse ได้ง่าย แต่ยังต้องตรวจเองว่า field ครบหรือไม่ Structured output จะช่วยให้เรากำหนด schema ที่ต้องการ และให้ SDK ช่วย parse กลับมาเป็น object ที่ใช้งานได้ทันที


In [ ]:
class TripPlan(BaseModel):
    destination: str = Field(description="จังหวัดหรือสถานที่หลัก")
    duration_days: int = Field(description="จำนวนวัน")
    highlights: list[str] = Field(description="กิจกรรมหรือไฮไลต์สำคัญ")
    estimated_budget_thb: int = Field(description="งบประมาณโดยประมาณ")
    tips: list[str] = Field(description="คำแนะนำเพิ่มเติม")


In [ ]:
structured_messages = [
    {"role": "system", "content": "คุณเป็นผู้ช่วยวางแผนท่องเที่ยวแบบกระชับและเป็นภาษาไทย"},
    {"role": "user", "content": "ช่วยจัดทริปเที่ยวน่าน 3 วัน 2 คืน สำหรับครอบครัว งบประมาณ 12000 บาท"},
]

structured_response = client.beta.chat.completions.parse(
    model=OPENAI_MODEL,
    messages=structured_messages,
    response_format=TripPlan,
    temperature=0.3,
)

trip_plan = structured_response.choices[0].message.parsed
trip_plan

TripPlan(destination='น่าน', duration_days=3, highlights=['เยี่ยมชมวัดพระธาตุแช่แห้ง', 'เดินป่าในอุทยานแห่งชาติศรีสว่าง', 'ชมวิวทะเลสาบห้วยขา', 'สำรวจตลาดน่านและชิมอาหารท้องถิ่น', 'พักที่รีสอร์ทบรรยากาศธรรมชาติ'], estimated_budget_thb=12000, tips=['เช็คสภาพอากาศและเตรียมเสื้อผ้าให้เหมาะสม', 'จองที่พักล่วงหน้าเพื่อความสะดวก', 'ใช้บริการรถเช่า/รถตู้เพื่อความคล่องตัว', 'พกน้ำดื่มและของว่างในระหว่างเดินป่า', 'เคารพวัฒนธรรมและศิลปะท้องถิ่น'])

In [ ]:
pretty_print_json(trip_plan.model_dump())

{
  "destination": "น่าน",
  "duration_days": 3,
  "highlights": [
    "เยี่ยมชมวัดพระธาตุแช่แห้ง",
    "เดินป่าในอุทยานแห่งชาติศรีสว่าง",
    "ชมวิวทะเลสาบห้วยขา",
    "สำรวจตลาดน่านและชิมอาหารท้องถิ่น",
    "พักที่รีสอร์ทบรรยากาศธรรมชาติ"
  ],
  "estimated_budget_thb": 12000,
  "tips": [
    "เช็คสภาพอากาศและเตรียมเสื้อผ้าให้เหมาะสม",
    "จองที่พักล่วงหน้าเพื่อความสะดวก",
    "ใช้บริการรถเช่า/รถตู้เพื่อความคล่องตัว",
    "พกน้ำดื่มและของว่างในระหว่างเดินป่า",
    "เคารพวัฒนธรรมและศิลปะท้องถิ่น"
  ]
}


## 6) Tool Calling แบบพื้นฐาน

แนวคิดของ tool calling คือให้โมเดลเลือกว่าจะ "เรียกฟังก์ชัน" ไหนก่อนตอบ แทนที่จะเดาข้อมูลทั้งหมดเอง

ตัวอย่างนี้ใช้ 3 tools แบบง่าย ๆ:
- `get_current_bangkok_time` ดูเวลาปัจจุบันในไทย
- `multiply_numbers` คำนวณตัวเลขง่าย ๆ
- `get_llm_term_definition` คืนคำอธิบายศัพท์ LLM สั้น ๆ จาก glossary ที่เราเตรียมไว้

จุดสำคัญคือ model จะไม่รัน Python เองโดยตรง แต่จะส่งคำขอเรียก tool พร้อม arguments มาให้เรา แล้วโค้ดฝั่ง Python ค่อย execute ให้


In [ ]:
from typing import Dict


def safe_json_dumps(obj: Any) -> str:
    return json.dumps(obj, ensure_ascii=False)


def assistant_to_message_dict(msg: Any) -> Dict[str, Any]:
    out: Dict[str, Any] = {"role": "assistant"}

    if getattr(msg, "content", None) is not None:
        out["content"] = msg.content

    tool_calls = getattr(msg, "tool_calls", None)
    if tool_calls:
        out["tool_calls"] = []
        for tc in tool_calls:
            out["tool_calls"].append(
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments,
                    },
                }
            )

    return out


def get_current_bangkok_time() -> dict:
    now_th = datetime.now(ZoneInfo("Asia/Bangkok"))
    return {
        "timezone": "Asia/Bangkok",
        "current_time": now_th.strftime("%Y-%m-%d %H:%M:%S"),
    }


def multiply_numbers(a: float, b: float) -> dict:
    return {
        "a": a,
        "b": b,
        "result": a * b,
    }


def get_llm_term_definition(term: str) -> dict:
    glossary = {
        "rag": "RAG ย่อมาจาก Retrieval-Augmented Generation คือการดึงข้อมูลภายนอกมาเสริมก่อนให้โมเดลตอบ",
        "embedding": "Embedding คือการแปลงข้อมูลให้เป็นเวกเตอร์ตัวเลขเพื่อใช้วัดความคล้ายหรือค้นคืนข้อมูล",
        "token": "Token คือหน่วยย่อยของข้อความที่โมเดลใช้ประมวลผลและคิดค่าใช้งาน",
    }
    key = term.strip().lower()
    return {
        "term": term,
        "definition": glossary.get(key, f"ยังไม่มีคำอธิบายสำหรับคำว่า {term}"),
    }


basic_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_bangkok_time",
            "description": "ดูเวลาปัจจุบันในประเทศไทย",
            "parameters": {
                "type": "object",
                "properties": {},
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "multiply_numbers",
            "description": "คูณตัวเลข 2 จำนวน",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "ตัวเลขตัวแรก"},
                    "b": {"type": "number", "description": "ตัวเลขตัวที่สอง"},
                },
                "required": ["a", "b"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_llm_term_definition",
            "description": "คืนคำอธิบายศัพท์พื้นฐานด้าน LLM จาก glossary ภายในโน้ตบุ๊ก",
            "parameters": {
                "type": "object",
                "properties": {
                    "term": {"type": "string", "description": "คำศัพท์ที่ต้องการอธิบาย"}
                },
                "required": ["term"],
            },
        },
    },
]

basic_available_functions = {
    "get_current_bangkok_time": get_current_bangkok_time,
    "multiply_numbers": multiply_numbers,
    "get_llm_term_definition": get_llm_term_definition,
}


In [ ]:
basic_tool_messages = [
    {
        "role": "system",
        "content": "คุณเป็นผู้ช่วยสอน LLM ภาษาไทย ถ้าต้องใช้ข้อมูลจาก tool ให้เรียกใช้ tool ก่อนตอบเสมอ",
    },
    {
        "role": "user",
        "content": "ตอนนี้ที่ไทยเวลาอะไร ช่วยคูณ 24 กับ 7 และอธิบายคำว่า RAG แบบสั้น ๆ",
    },
]

basic_response_original = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=basic_tool_messages,
    tools=basic_tools,
    tool_choice="auto",
    temperature=0,
)

basic_message = basic_response_original.choices[0].message
basic_tool_traces = []

display(basic_response_original)
print('='*50)

while basic_message.tool_calls:
    basic_tool_messages.append(assistant_to_message_dict(basic_message))

    for tool_call in basic_message.tool_calls:
        tool_name = tool_call.function.name
        tool_args = json.loads(tool_call.function.arguments or "{}")
        tool_result = basic_available_functions[tool_name](**tool_args)

        basic_tool_traces.append(
            {
                "tool": tool_name,
                "arguments": tool_args,
                "result": tool_result,
            }
        )

        basic_tool_messages.append(
            {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": safe_json_dumps(tool_result),
            }
        )

    basic_response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=basic_tool_messages,
        tools=basic_tools,
        tool_choice="auto",
        temperature=0,
    )
    basic_message = basic_response.choices[0].message

pretty_print_json(basic_tool_messages)
print("\nFinal answer:\n")
print(basic_message.content)


ChatCompletion(id='chatcmpl-fecfa4c9-bed3-449e-b533-7162ec74e0f4', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='fc_91ba7165-cf65-454e-8bec-41d86b312495', function=Function(arguments='{}', name='get_current_bangkok_time'), type='function')], reasoning='We need to get current Bangkok time, multiply 24*7, and get definition of term "RAG". Must call tools before answering. So three calls: get_current_bangkok_time, multiply_numbers, get_llm_term_definition.'))], created=1774328655, model='openai/gpt-oss-120b', object='chat.completion', service_tier='on_demand', system_fingerprint='fp_60e4b492db', usage=CompletionUsage(completion_tokens=73, prompt_tokens=273, total_tokens=346, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_token

[
  {
    "role": "system",
    "content": "คุณเป็นผู้ช่วยสอน LLM ภาษาไทย ถ้าต้องใช้ข้อมูลจาก tool ให้เรียกใช้ tool ก่อนตอบเสมอ"
  },
  {
    "role": "user",
    "content": "ตอนนี้ที่ไทยเวลาอะไร ช่วยคูณ 24 กับ 7 และอธิบายคำว่า RAG แบบสั้น ๆ"
  },
  {
    "role": "assistant",
    "tool_calls": [
      {
        "id": "fc_91ba7165-cf65-454e-8bec-41d86b312495",
        "type": "function",
        "function": {
          "name": "get_current_bangkok_time",
          "arguments": "{}"
        }
      }
    ]
  },
  {
    "role": "tool",
    "tool_call_id": "fc_91ba7165-cf65-454e-8bec-41d86b312495",
    "content": "{\"timezone\": \"Asia/Bangkok\", \"current_time\": \"2026-03-24 12:04:15\"}"
  },
  {
    "role": "assistant",
    "tool_calls": [
      {
        "id": "fc_0d133ae6-9e77-40a6-a33b-20be7b1c648f",
        "type": "function",
        "function": {
          "name": "multiply_numbers",
          "arguments": "{\"a\":24,\"b\":7}"
        }
      }
    ]
  },
  {
    "role": "tool",
  

## 7) Deep Research ด้วย DuckDuckGo

หลังจากเห็นภาพ tool calling แบบพื้นฐานแล้ว ลองขยับไปเป็น workflow ที่ซับซ้อนขึ้น โดยให้โมเดลวางแผน ค้นข้อมูล ประเมินคุณภาพ และสรุปโน้ตก่อนตอบจริง

flow ของตัวอย่างนี้มีหลาย tools และให้โมเดลเรียกตามลำดับงานวิจัย:
1. `plan_research` วางแผนว่าจะค้นอะไรบ้าง
2. `duckduckgo_search` ค้นข้อมูลจากเว็บ
3. `evaluate_search_results` ประเมินว่าผลค้นหาใช้ตอบได้แค่ไหนและยังขาดอะไร
4. `summarize_research_notes` รวมโน้ตทั้งหมดให้พร้อมสำหรับ final answer

ตัวอย่างนี้ช่วยให้เห็นว่า tool calling ไม่ได้มีแค่ "search อย่างเดียว" แต่สามารถออกแบบเป็น workflow หลายขั้นตอนแบบ deep research ได้


In [ ]:
# =========================================================
# Helpers
# =========================================================
from typing import Optional

def safe_json_dumps(obj: Any) -> str:
    return json.dumps(obj, ensure_ascii=False)


def pretty_print_json(data: Any) -> None:
    print(json.dumps(data, ensure_ascii=False, indent=2))


def assistant_to_message_dict(msg: Any) -> Dict[str, Any]:
    out: Dict[str, Any] = {"role": "assistant"}

    if getattr(msg, "content", None) is not None:
        out["content"] = msg.content

    tool_calls = getattr(msg, "tool_calls", None)
    if tool_calls:
        out["tool_calls"] = []
        for tc in tool_calls:
            out["tool_calls"].append(
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments,
                    },
                }
            )

    return out


def parse_json_response(content: str, fallback: Optional[dict] = None) -> dict:
    if fallback is None:
        fallback = {}
    try:
        parsed = json.loads(content)
        if isinstance(parsed, dict):
            return parsed
        return fallback
    except Exception:
        return fallback


def normalize_search_payload(search_results_json: str) -> Dict[str, Any]:
    """
    รองรับทั้ง:
    1) dict -> {"query": "...", "results": [...]}
    2) list -> [{...}, {...}]
    """
    payload = json.loads(search_results_json)

    if isinstance(payload, dict):
        query = payload.get("query")
        results = payload.get("results", [])
    elif isinstance(payload, list):
        query = None
        results = payload
    else:
        query = None
        results = []

    normalized_results = []
    for item in results:
        if not isinstance(item, dict):
            continue
        normalized_results.append(
            {
                "title": item.get("title") or "Untitled",
                "snippet": item.get("body") or item.get("snippet") or "",
                "url": item.get("href") or item.get("url") or "",
            }
        )

    return {"query": query, "results": normalized_results}


def truncate_text(text: str, max_len: int = 220) -> str:
    text = (text or "").strip().replace("\n", " ")
    if len(text) <= max_len:
        return text
    return text[: max_len - 3] + "..."


def simplify_search_results_payload(search_results_json: str, max_items: int = 5) -> str:
    """
    แปลงผลค้นหาให้เป็น payload แบบเบาและปลอดภัยสำหรับส่งเข้า LLM/tool
    """
    normalized = normalize_search_payload(search_results_json)
    simple_results = []

    for item in normalized["results"][:max_items]:
        simple_results.append(
            {
                "title": truncate_text(item.get("title", ""), 120),
                "snippet": truncate_text(item.get("snippet", ""), 180),
                "url": item.get("url", ""),
            }
        )

    compact_payload = {
        "query": normalized.get("query"),
        "results": simple_results,
    }
    return json.dumps(compact_payload, ensure_ascii=False)

In [ ]:
import json
from ddgs import DDGS
from typing import Any, Dict, List


# =========================================================
# LLM-backed tool functions
# =========================================================

def plan_research(topic: str, goal: str = "สรุปข้อมูลที่เชื่อถือได้") -> dict:
    """
    ใช้ LLM ช่วยวางแผนการค้นข้อมูลแบบง่าย
    """
    planning_system_prompt = """
คุณเป็น research planner ภาษาไทย
ช่วยวางแผนค้นข้อมูลแบบสั้นและใช้งานได้จริง

ตอบเป็น JSON เท่านั้น ตาม schema นี้:
{
  "topic": "string",
  "goal": "string",
  "search_queries": ["string"],
  "evaluation_criteria": ["string"],
  "notes": "string"
}

ข้อกำหนด:
- search_queries เป็นคำค้นที่เอาไปค้นต่อได้จริง 1-2 ข้อ
- evaluation_criteria เป็นเกณฑ์ง่าย ๆ สำหรับเช็กคุณภาพข้อมูล 1 ข้อ
- notes เป็นคำแนะนำสั้น ๆ
- ห้ามตอบข้อความอื่นนอก JSON
"""

    planning_user_prompt = f"""
หัวข้อ: {topic}
เป้าหมาย: {goal}
"""

    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        temperature=0.2,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": planning_system_prompt},
            {"role": "user", "content": planning_user_prompt},
        ],
    )

    parsed = parse_json_response(
        response.choices[0].message.content,
        fallback={
            "topic": topic,
            "goal": goal,
            "search_queries": [
                f"{topic} overview",
                f"{topic} latest news",
                f"{topic} expert analysis",
            ],
            "evaluation_criteria": [
                "ข้อมูลใหม่พอ",
                "มีหลายแหล่งอ้างอิง",
                "มีตัวอย่างหรือข้อสรุปที่นำไปใช้ได้",
            ],
            "notes": "เริ่มจากภาพรวมก่อน แล้วค่อยค้นเพิ่มในมุมที่ยังขาด",
        },
    )

    parsed.setdefault("topic", topic)
    parsed.setdefault("goal", goal)
    parsed.setdefault("search_queries", [
        f"{topic} overview",
        f"{topic} latest news",
        f"{topic} expert analysis",
    ])
    parsed.setdefault("evaluation_criteria", [
        "ข้อมูลใหม่พอ",
        "มีหลายแหล่งอ้างอิง",
        "มีตัวอย่างหรือข้อสรุปที่นำไปใช้ได้",
    ])
    parsed.setdefault("notes", "เริ่มจากภาพรวมก่อน แล้วค่อยค้นเพิ่มในมุมที่ยังขาด")

    return parsed


def duckduckgo_search(query: str, max_results: int = 5) -> dict:
    """
    ค้นเว็บจริงด้วย DDGS แล้วคืนค่าใน format ที่ tool อื่นใช้ต่อได้
    จำกัดผลลัพธ์ไม่เกิน 5 เพื่อให้ payload เล็กและเสถียร
    """
    max_results = min(max_results or 5, 5)
    results = []

    with DDGS() as ddgs:
        search_iter = ddgs.text(
            query,
            max_results=max_results,
        )

        for item in search_iter:
            results.append(
                {
                    "title": item.get("title", ""),
                    "body": truncate_text(item.get("body", "") or item.get("snippet", ""), 300),
                    "href": item.get("href", "") or item.get("url", ""),
                }
            )

    return {
        "query": query,
        "results": results,
    }


def evaluate_search_results(topic: str, search_results_json: str, search_round: int = 1) -> dict:
    """
    ประเมินผลค้นหาแบบง่ายขึ้น
    - payload เบา
    - เกณฑ์ผ่านง่าย
    - ถ้าค้นเกิน 3 รอบ ให้ผ่านเลย
    """
    normalized = normalize_search_payload(search_results_json)
    query = normalized["query"]
    results = normalized["results"][:5]

    real_sources = [r["url"] for r in results if r.get("url")]
    unique_sources = list(dict.fromkeys(real_sources))

    # ผ่านอัตโนมัติถ้าค้นเกิน 3 รอบ
    auto_pass = search_round >= 3

    # heuristic แบบง่าย
    easy_pass = auto_pass or len(results) >= 2 or len(unique_sources) >= 2

    eval_system_prompt = """
คุณเป็นผู้ประเมินคุณภาพงานวิจัยภาษาไทย
ให้ประเมินแบบง่ายและไม่เข้มงวดเกินไป

ตอบเป็น JSON เท่านั้น ตาม schema นี้:
{
  "topic": "string",
  "query": "string or null",
  "insights": ["string"],
  "sources": ["string"],
  "quality_check": {
    "result_count": 0,
    "has_multiple_sources": true,
    "is_sufficient": true,
    "needs_more_search": false,
    "reason": "string",
    "missing_angles": ["string"],
    "suggested_queries": ["string"]
  }
}

กติกา:
- ถ้ามีผลลัพธ์อย่างน้อย 2 รายการ หรือมีแหล่งข้อมูลจริงอย่างน้อย 2 แหล่ง ให้ถือว่าเพียงพอได้
- ถ้ายังไม่พอ suggested_queries ให้เสนอไม่เกิน 2 ข้อ
- insights เอาแบบสั้น กระชับ
- ห้ามตอบข้อความอื่นนอก JSON
""".strip()

    eval_user_prompt = f"""
หัวข้อ: {topic}
คำค้นปัจจุบัน: {query}
รอบการค้น: {search_round}

ผลการค้นหา:
{json.dumps(results, ensure_ascii=False, indent=2)}
""".strip()

    try:
        response = client.chat.completions.create(
            model=OPENAI_MODEL,
            temperature=0,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": eval_system_prompt},
                {"role": "user", "content": eval_user_prompt},
            ],
        )

        parsed = parse_json_response(
            response.choices[0].message.content,
            fallback={},
        )
    except Exception:
        parsed = {}

    if not isinstance(parsed, dict):
        parsed = {}

    parsed.setdefault("topic", topic)
    parsed.setdefault("query", query)
    parsed.setdefault(
        "insights",
        [
            item["title"]
            for item in results[:3]
            if item.get("title")
        ],
    )
    parsed.setdefault("sources", unique_sources)

    qc = parsed.setdefault("quality_check", {})
    qc.setdefault("result_count", len(results))
    qc.setdefault("has_multiple_sources", len(unique_sources) >= 2)

    # override ให้ผ่านง่าย
    if easy_pass:
        qc["is_sufficient"] = True
        qc["needs_more_search"] = False
        qc["reason"] = (
            "ผ่านเกณฑ์แบบง่าย: มีผลลัพธ์/แหล่งข้อมูลเพียงพอ"
            if not auto_pass
            else "ค้นมาครบ 3 รอบแล้ว ให้ถือว่าข้อมูลพอสำหรับสรุป"
        )
        qc["missing_angles"] = qc.get("missing_angles", [])
        qc["suggested_queries"] = []
    else:
        qc.setdefault("is_sufficient", False)
        qc.setdefault("needs_more_search", True)
        qc.setdefault("reason", "ผลลัพธ์ยังน้อย ควรค้นเพิ่มอีก 1 รอบ")
        qc.setdefault("missing_angles", ["ข้อมูลเปรียบเทียบเพิ่มเติม"])
        qc.setdefault("suggested_queries", [f"{topic} comparison"])

    return parsed


def summarize_research_notes(topic: str, evaluated_notes_json: str) -> dict:
    """
    ใช้ LLM สรุปโน้ตที่ผ่านการประเมินแล้วให้พร้อมสำหรับ final answer
    """
    notes = json.loads(evaluated_notes_json)
    if not isinstance(notes, dict):
        notes = {}

    summary_system_prompt = """
คุณเป็น research synthesizer ภาษาไทย
หน้าที่คือสรุปผลวิจัยจากโน้ตที่ผ่านการประเมินแล้ว ให้พร้อมสำหรับนำไปตอบ final answer

ให้ตอบเป็น JSON เท่านั้น ตาม schema นี้:
{
  "topic": "string",
  "draft_summary": "string",
  "key_points": ["string"],
  "source_count": 0,
  "source_urls": ["string"],
  "next_action": "string"
}

ข้อกำหนด:
- draft_summary ต้องเป็นสรุปภาพรวมภาษาไทยแบบกระชับ
- key_points ต้องเป็นประเด็นสำคัญที่เอาไปใช้ตอบต่อได้จริง
- next_action ให้ระบุว่าเพียงพอจะตอบ final แล้ว หรือควรค้นเพิ่ม
- source_urls ให้ดึงจาก notes ถ้ามี
- ห้ามตอบข้อความอื่นนอก JSON
""".strip()

    summary_user_prompt = f"""
หัวข้อ: {topic}

โน้ตที่ผ่านการประเมิน:
{json.dumps(notes, ensure_ascii=False, indent=2)}
""".strip()

    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        temperature=0.2,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": summary_system_prompt},
            {"role": "user", "content": summary_user_prompt},
        ],
    )

    parsed = parse_json_response(
        response.choices[0].message.content,
        fallback={
            "topic": topic,
            "draft_summary": f"สรุปข้อมูลเกี่ยวกับ {topic}",
            "key_points": notes.get("insights", []),
            "source_count": len(notes.get("sources", [])),
            "source_urls": notes.get("sources", []),
            "next_action": "พร้อมให้โมเดลสรุป final answer ภาษาไทย",
        },
    )

    parsed.setdefault("topic", topic)
    parsed.setdefault("draft_summary", f"สรุปข้อมูลเกี่ยวกับ {topic}")
    parsed.setdefault("key_points", notes.get("insights", []))
    parsed.setdefault("source_count", len(notes.get("sources", [])))
    parsed.setdefault("source_urls", notes.get("sources", []))
    parsed.setdefault("next_action", "พร้อมให้โมเดลสรุป final answer ภาษาไทย")

    return parsed



# =========================================================
# Tool schema
# =========================================================

research_tools = [
    {
        "type": "function",
        "function": {
            "name": "plan_research",
            "description": "ใช้ LLM วางแผนการค้นข้อมูล กำหนดคำค้น มุมวิจัย และเกณฑ์ประเมินคุณภาพข้อมูล",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {
                        "type": "string",
                        "description": "หัวข้อที่ต้องการวิจัย",
                    },
                    "goal": {
                        "type": "string",
                        "description": "เป้าหมายของงานวิจัย",
                    },
                },
                "required": ["topic"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "duckduckgo_search",
            "description": "ค้นหาข้อมูลจากเว็บด้วย DDGS/DuckDuckGo เพื่อใช้ตอบคำถามที่ต้องการข้อมูลล่าสุดหรือหลายแหล่ง",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "คำค้นที่ใช้ค้นหา",
                    },
                    "max_results": {
                        "type": "integer",
                        "description": "จำนวนผลลัพธ์สูงสุด",
                        "default": 5,
                    },
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "evaluate_search_results",
            "description": "ใช้ LLM ประเมินผลค้นหาว่าข้อมูลเพียงพอหรือยัง ดึง insight สำคัญ และเสนอ query สำหรับค้นต่อถ้ายังไม่พอ",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {
                        "type": "string",
                        "description": "หัวข้อที่กำลังวิจัย",
                    },
                    "search_results_json": {
                        "type": "string",
                        "description": "ผลลัพธ์จาก search tool ในรูปแบบ JSON string",
                    },
                    "search_round": {
                        "type": "integer",
                        "description": "รอบการค้นปัจจุบัน",
                        "default": 1,
                    },
                },
                "required": ["topic", "search_results_json"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "summarize_research_notes",
            "description": "ใช้ LLM สังเคราะห์โน้ตที่ผ่านการประเมินแล้วให้พร้อมสำหรับ final answer",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {
                        "type": "string",
                        "description": "หัวข้อที่กำลังวิจัย",
                    },
                    "evaluated_notes_json": {
                        "type": "string",
                        "description": "โน้ตจาก evaluate tool ในรูปแบบ JSON string",
                    },
                },
                "required": ["topic", "evaluated_notes_json"],
            },
        },
    },
]



# =========================================================
# Available functions
# =========================================================

available_functions = {
    "plan_research": plan_research,
    "duckduckgo_search": duckduckgo_search,
    "evaluate_search_results": evaluate_search_results,
    "summarize_research_notes": summarize_research_notes,
}

In [ ]:
# =========================================================
# Agent prompts
# =========================================================

research_system_prompt = """\
คุณเป็น research assistant ภาษาไทย สำหรับคำถามที่ต้องใช้ข้อมูลล่าสุด

ให้ทำงานเป็นขั้นตอน:
1) plan_research เพื่อวางแผน
2) duckduckgo_search เพื่อค้น
3) evaluate_search_results เพื่อประเมินว่าข้อมูลพอหรือยัง

กติกา:
- ให้เริ่มจาก plan_research ก่อนเสมอ
- หลังจากได้แผนแล้ว ให้เลือก search_queries ที่เหมาะสมไปค้น
- เมื่อค้นแล้วต้องเรียก evaluate_search_results เพื่อตัดสิน
- ถ้า evaluate บอกว่ายังไม่พอ ให้ค้นเพิ่มโดยใช้ suggested_queries หรือมุมค้นที่ยังขาด
- เมื่อข้อมูลพอแล้ว จึงเรียก summarize_research_notes
- เมื่อใช้ summarize_research_notes แล้วให้ตอบทันที
- เมื่อได้ summary แล้ว ค่อยตอบ final answer ภาษาไทย
- final answer ควรสรุปอย่างระมัดระวัง ไม่อวดอ้างเกินหลักฐานที่มี
"""

research_messages = [
    {"role": "system", "content": research_system_prompt},
    {"role": "user","content": "model LLM (MMLU) ที่เก่งที่สุด ในปี 2026",},
]

tool_request = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=research_messages,
    tools=research_tools,
    tool_choice="auto",
    temperature=0,
)

assistant_message = tool_request.choices[0].message
assistant_message

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='fc_856628b2-b7e0-4159-bbec-7798eb4172fd', function=Function(arguments='{"goal":"ค้นหาโมเดล LLM ที่มีคะแนน MMLU สูงที่สุดในปี 2026","topic":"LLM MMLU best model 2026"}', name='plan_research'), type='function')], reasoning='The user asks: "model LLM (MMLU) ที่เก่งที่สุด ในปี 2026" meaning "the best LLM model (MMLU) in 2026". They want the best LLM model for MMLU in 2026. We need up-to-date info. We must follow the instruction: start with plan_research.')

In [ ]:
# OPENAI_MODEL =  "openai/gpt-oss-120b"
OPENAI_MODEL =  "openai/gpt-oss-20b"

In [ ]:
tool_traces = []
current_message = assistant_message
step = 1


while current_message.tool_calls and step <= 12:
    print("\n" + "=" * 80)
    print(f"STEP {step}: model requested tools")
    print("=" * 80)

    research_messages.append(assistant_to_message_dict(current_message))

    for i, tool_call in enumerate(current_message.tool_calls, start=1):
        tool_name = tool_call.function.name
        tool_args = json.loads(tool_call.function.arguments)

        print(f"\n[{step}.{i}] Executing: {tool_name}")
        print(f"[{step}.{i}] Arguments:")
        pretty_print_json(tool_args)

        if tool_name not in available_functions:
            raise ValueError(f"Unknown tool called: {tool_name}")

        try:
            tool_result = available_functions[tool_name](**tool_args)
            status = "success"
        except Exception as e:
            tool_result = {
                "error": str(e),
                "tool": tool_name,
                "arguments": tool_args,
            }
            status = f"error: {e}"

        print(f"[{step}.{i}] Result ({status}):")
        pretty_print_json(tool_result)

        tool_traces.append(
            {
                "tool": tool_name,
                "arguments": tool_args,
                "result": tool_result,
            }
        )

        research_messages.append(
            {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": safe_json_dumps(tool_result),
            }
        )

    next_response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=research_messages,
        tools=research_tools,
        tool_choice="auto",
        temperature=0.3,
    )

    current_message = next_response.choices[0].message
    step += 1

print("\n" + "=" * 80)
print("ALL TOOL TRACES")
print("=" * 80)



STEP 1: model requested tools

[1.1] Executing: plan_research
[1.1] Arguments:
{
  "goal": "ค้นหาโมเดล LLM ที่มีคะแนน MMLU สูงที่สุดในปี 2026",
  "topic": "LLM MMLU best model 2026"
}
[1.1] Result (success):
{
  "topic": "LLM MMLU best model 2026",
  "goal": "ค้นหาโมเดล LLM ที่มีคะแนน MMLU สูงที่สุดในปี 2026",
  "search_queries": [
    "LLM MMLU score 2026 best model",
    "2026 MMLU benchmark top LLM"
  ],
  "evaluation_criteria": [
    "ตรวจสอบคะแนน MMLU สูงสุดจากรายงานล่าสุด"
  ],
  "notes": "ใช้แหล่งข้อมูลเช่น arXiv, ACL Anthology, หรือเว็บไซต์ของผู้พัฒนาโมเดล"
}

STEP 2: model requested tools

[2.1] Executing: duckduckgo_search
[2.1] Arguments:
{
  "max_results": 10,
  "query": "LLM MMLU score 2026 best model"
}
[2.1] Result (success):
{
  "query": "LLM MMLU score 2026 best model",
  "results": [
    {
      "title": "Best LLM Leaderboard 2026 | AI Model Rankings, Benchmarks & Pricing",
      "body": "The definitiveLLMleaderboard — ranking thebestAImodelsincluding Claude, GPT, Ge

In [ ]:
pretty_print_json(tool_traces)

[
  {
    "tool": "plan_research",
    "arguments": {
      "goal": "ค้นหาโมเดล LLM ที่มีคะแนน MMLU สูงที่สุดในปี 2026",
      "topic": "LLM MMLU best model 2026"
    },
    "result": {
      "topic": "LLM MMLU best model 2026",
      "goal": "ค้นหาโมเดล LLM ที่มีคะแนน MMLU สูงที่สุดในปี 2026",
      "search_queries": [
        "LLM MMLU score 2026 best model",
        "2026 MMLU benchmark top LLM"
      ],
      "evaluation_criteria": [
        "ตรวจสอบคะแนน MMLU สูงสุดจากรายงานล่าสุด"
      ],
      "notes": "ใช้แหล่งข้อมูลเช่น arXiv, ACL Anthology, หรือเว็บไซต์ของผู้พัฒนาโมเดล"
    }
  },
  {
    "tool": "duckduckgo_search",
    "arguments": {
      "max_results": 10,
      "query": "LLM MMLU score 2026 best model"
    },
    "result": {
      "query": "LLM MMLU score 2026 best model",
      "results": [
        {
          "title": "Best LLM Leaderboard 2026 | AI Model Rankings, Benchmarks & Pricing",
          "body": "The definitiveLLMleaderboard — ranking thebestAImodelsincludin

In [ ]:
research_messages.append(current_message.model_dump())
final_research_answer = current_message.content

print(final_research_answer)

**โมเดล LLM ที่มีคะแนน MMLU สูงที่สุดในปี 2026**

| โมเดล | คะแนน MMLU (ประมาณ) | ความเด่นเพิ่มเติม |
|--------|----------------------|--------------------|
| **Claude 3.5** | ~91.7 % | เป็นโมเดลชั้นนำในตารางคะแนน MMLU 2026 โดยมีคะแนนสูงกว่ารุ่นอื่น ๆ ในหลาย ๆ ด้านของความเข้าใจและการแก้ปัญหา |
| GPT‑4.5 | ~90 % | ยังคงมีความแข็งแกร่งในงานเขียนโค้ดและการแก้ปัญหาทางตรรกะ แม้คะแนน MMLU จะต่ำกว่า Claude 3.5 เล็กน้อย |
| โมเดลอื่น ๆ (เช่น GLM‑5, DeepSeek V3.2 ฯลฯ) | 80‑90 % | มีคะแนนที่แข่งขันได้ แต่ยังไม่เกิน Claude 3.5 ในปี 2026 |

**แหล่งข้อมูลที่ใช้ตรวจสอบ**

1. **Onyx LLM Leaderboard** – รายงานอันดับโมเดลโดยรวม รวมถึงคะแนน MMLU 2026  
   <https://onyx.app/llm-leaderboard>
2. **LLM‑Stats Leaderboard** – แสดงคะแนน MMLU ของโมเดลหลายรุ่นในปี 2026  
   <https://llm-stats.com/leaderboards/llm-...>
3. **PricePerToken MMLU Leaderboard** – รายงานคะแนน MMLU ล่าสุด (อัปเดตถึง 10 มีนาคม 2026)  
   <https://pricepertoken.com/leaderboards/benchmark/mmlu>

**ข้อสังเกต**

- คะแนน MMLU ของ Claude 3.5 อ

In [ ]:
display(Markdown(final_research_answer))

**โมเดล LLM ที่มีคะแนน MMLU สูงที่สุดในปี 2026**

| โมเดล | คะแนน MMLU (ประมาณ) | ความเด่นเพิ่มเติม |
|--------|----------------------|--------------------|
| **Claude 3.5** | ~91.7 % | เป็นโมเดลชั้นนำในตารางคะแนน MMLU 2026 โดยมีคะแนนสูงกว่ารุ่นอื่น ๆ ในหลาย ๆ ด้านของความเข้าใจและการแก้ปัญหา |
| GPT‑4.5 | ~90 % | ยังคงมีความแข็งแกร่งในงานเขียนโค้ดและการแก้ปัญหาทางตรรกะ แม้คะแนน MMLU จะต่ำกว่า Claude 3.5 เล็กน้อย |
| โมเดลอื่น ๆ (เช่น GLM‑5, DeepSeek V3.2 ฯลฯ) | 80‑90 % | มีคะแนนที่แข่งขันได้ แต่ยังไม่เกิน Claude 3.5 ในปี 2026 |

**แหล่งข้อมูลที่ใช้ตรวจสอบ**

1. **Onyx LLM Leaderboard** – รายงานอันดับโมเดลโดยรวม รวมถึงคะแนน MMLU 2026  
   <https://onyx.app/llm-leaderboard>
2. **LLM‑Stats Leaderboard** – แสดงคะแนน MMLU ของโมเดลหลายรุ่นในปี 2026  
   <https://llm-stats.com/leaderboards/llm-...>
3. **PricePerToken MMLU Leaderboard** – รายงานคะแนน MMLU ล่าสุด (อัปเดตถึง 10 มีนาคม 2026)  
   <https://pricepertoken.com/leaderboards/benchmark/mmlu>

**ข้อสังเกต**

- คะแนน MMLU ของ Claude 3.5 อยู่ในระดับสูงสุดโดยมีความแตกต่างเล็กน้อยจาก GPT‑4.5  
- ความแตกต่างของคะแนนอาจเกิดจากวิธีการฝึกและขนาดโมเดล รวมถึงการปรับแต่งพารามิเตอร์เฉพาะสำหรับงานด้านความเข้าใจภาษา  
- หากต้องการข้อมูลละเอียดเพิ่มเติม เช่น วิธีการประเมินหรือรายละเอียดของชุดทดสอบ MMLU ควรตรวจสอบเอกสารทางเทคนิคของแต่ละโมเดลหรือรายงานการทดสอบจากผู้พัฒนา

**สรุป**  
ในปี 2026 โมเดล Claude 3.5 ถือเป็นโมเดล LLM ที่มีคะแนน MMLU สูงที่สุดตามข้อมูลล่าสุดที่มีอยู่ โดย GPT‑4.5 ยังเป็นตัวเลือกที่แข็งแกร่งในด้านการเขียนโค้ดและตรรกะ แต่คะแนน MMLU ของมันยังต่ำกว่า Claude 3.5 เล็กน้อย.

## 8) Advanced: OCR และ Vision ด้วย Gemini

แม้ notebook นี้ใช้ `openai` library เป็นแกนหลัก แต่เราสามารถเรียก Gemini ผ่าน OpenAI-compatible endpoint ได้เช่นกัน

ตัวอย่างด้านล่างใช้ไฟล์ PDF ที่อัปโหลดไว้ใน Colab แล้วแปลงเป็น base64 เพื่อส่งเข้าโมเดล


In [ ]:
from google.colab import files

uploaded = files.upload()
pdf_path = next(iter(uploaded.keys()))


In [ ]:
!gdown 1Jvm2QvTij7W5epqXNveDkoOABfoNkfC5

Downloading...
From: https://drive.google.com/uc?id=1Jvm2QvTij7W5epqXNveDkoOABfoNkfC5
To: /content/example_doc.pdf
100% 1.06M/1.06M [00:00<00:00, 9.19MB/s]


In [ ]:
GEMINI_MODEL = "gemini-2.5-flash"

def build_gemini_client() -> OpenAI:
    return OpenAI(
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
        api_key=userdata.get("GEMINI_API_KEY"),
    )

gemini_client = build_gemini_client()

In [ ]:
pdf_path ='/content/example_doc.pdf'

with open(pdf_path, "rb") as f:
    pdf_base64 = base64.b64encode(f.read()).decode("utf-8")

print(f"Loaded file: {pdf_path}")

Loaded file: /content/example_doc.pdf


In [ ]:
summary_messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image_url",
                "image_url": {"url": f"data:application/pdf;base64,{pdf_base64}"},
            },
            {
                "type": "text",
                "text": "ช่วยสรุปเอกสารนี้เป็น bullet points ภาษาไทย โดยเน้นประเด็นสำคัญที่คนทั่วไปอ่านแล้วเข้าใจได้เร็ว",
            },
        ],
    }
]

summary_response = gemini_client.chat.completions.create(
    model=GEMINI_MODEL,
    messages=summary_messages,
)

print(summary_response.choices[0].message.content)

สรุปประเด็นสำคัญของเอกสารนี้เป็นภาษาไทยแบบ bullet points ดังนี้:

*   **เรื่อง:** ประกาศรายชื่อผู้มีสิทธิเข้าประเมินสมรรถนะ ครั้งที่ 1 สำหรับการสรรหาและเลือกสรรบุคคลเป็นลูกจ้างชั่วคราว (รายวัน) สังกัดสำนักงานสาธารณสุขอำเภอดำเนินสะดวก
*   **ประกาศ ณ วันที่:** 11 พฤศจิกายน 2567
*   **ตำแหน่งที่เปิดรับ:**
    *   **พยาบาลวิชาชีพ:** ไม่มีผู้สมัครที่ผ่านเกณฑ์ (ไม่มีผู้มีสิทธิเข้ารับการประเมิน)
    *   **นักวิชาการสาธารณสุข:** มีผู้มีสิทธิเข้ารับการประเมิน 3 ท่าน ได้แก่ นายชัยธวัช กรุดโกศล, นางสาวจิรัชญา บุญเพ็ญ, และ นางสาวณัฐกานต์ ทินโน
*   **กำหนดการประเมินสมรรถนะ (สำหรับตำแหน่งนักวิชาการสาธารณสุข):**
    *   **วัน:** 14 พฤศจิกายน 2567
    *   **เวลา:** 09.00 - 11.00 น. (รายงานตัวเวลา 08.30 น.)
    *   **สถานที่:** สำนักงานสาธารณสุขอำเภอดำเนินสะดวก
    *   **รูปแบบ:** สอบข้อเขียน (ความรู้ความสามารถทั่วไป 40 คะแนน, ความรู้เฉพาะตำแหน่ง 60 คะแนน)
*   **ข้อปฏิบัติและข้อห้ามในการเข้าสอบ:**
    *   ต้องนำบัตรประจำตัวประชาชนไปแสดงเพื่อยืนยันตัวตน
    *   ห้ามนำโทรศัพท์มือถือ, อุปกรณ์สื่อสาร, กล้อ

In [ ]:
ocr_messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image_url",
                "image_url": {"url": f"data:application/pdf;base64,{pdf_base64}"},
            },
            {
                "type": "text",
                "text": "อ่านข้อความจากเอกสารนี้และถอดออกมาเป็น Markdown ภาษาไทย โดยพยายามรักษาโครงสร้างหัวข้อเดิมให้มากที่สุด",
            },
        ],
    }
]

ocr_response = gemini_client.chat.completions.create(
    model=GEMINI_MODEL,
    messages=ocr_messages,
)

ocr_markdown = ocr_response.choices[0].message.content
display(Markdown(ocr_markdown))

# ประกาศสำนักงานสาธารณสุขอำเภอดำเนินสะดวก
**เรื่อง รายชื่อผู้มีสิทธิเข้าประเมินสมรรถนะ ครั้งที่ ๑ ในการรับสมัครบุคคลเพื่อสรรหาและเลือกสรรเป็นลูกจ้างชั่วคราว (รายวัน)**
********************************

ตามที่ สำนักงานสาธารณสุขอำเภอดำเนินสะดวก ได้ประกาศรับสมัครบุคคลเพื่อสรรหาและเลือกสรรเป็นลูกจ้างชั่วคราว (รายวัน) ตามหลักเกณฑ์ วิธีการและเงื่อนไขการจ่ายเงินบำรุง เพื่อเป็นค่าจ้างลูกจ้างชั่วคราวหรือลูกจ้างรายคาบของหน่วยบริการในสังกัดกระทรวงสาธารณสุข พ.ศ. ๒๕๔๕ ปฏิบัติราชการที่โรงพยาบาลส่งเสริมสุขภาพตำบลในสังกัด จำนวน ๔ อัตรา ประกอบด้วย ตำแหน่ง พยาบาลวิชาชีพ จำนวน ๒ อัตรา และตำแหน่ง นักวิชาการสาธารณสุข จำนวน ๒ อัตรา โดยให้ผู้ประสงค์จะสมัคร สามารถขอและยื่นใบสมัครด้วยตนเองที่สำนักงานสาธารณสุขอำเภอดำเนินสะดวก ตั้งแต่ วันที่ ๓๐ ตุลาคม ๒๕๖๗ ถึงวันที่ ๘ พฤศจิกายน ๒๕๖๗ ที่ผ่านมานั้น

บัดนี้ สำนักงานสาธารณสุขอำเภอดำเนินสะดวก จึงขอประกาศ “รายชื่อผู้มีสิทธิเข้าประเมินสมรรถนะ ครั้งที่ ๑ ในการรับสมัครบุคคลเพื่อสรรหาและเลือกสรรเป็นลูกจ้างชั่วคราว (รายวัน)” ดังบัญชีรายละเอียดแนบท้ายประกาศฉบับนี้ และให้ผู้มีรายชื่อ เข้ารับการประเมินสมรรถนะตามวันและเวลาที่กำหนด ต่อไป

จึงประกาศให้ทราบโดยทั่วกัน

ประกาศ ณ วันที่ ๑๑ พฤศจิกายน พ.ศ. ๒๕๖๗

(นายสิทธิชน จันทร์แพง)
สาธารณสุขอำเภอดำเนินสะดวก

---

## รายชื่อผู้มีสิทธิ์เข้ารับการประเมินสมรรถนะ ครั้งที่ ๑
### ตำแหน่ง พยาบาลวิชาชีพ
**แนบท้ายประกาศสำนักงานสาธารณสุขอำเภอดำเนินสะดวก ลงวันที่ ๑๑ พฤศจิกายน ๒๕๖๗**

| หมายเลขผู้สมัคร | ชื่อ-สกุล |
|---------------|---------|
| ไม่มีผู้สมัคร   |         |

---

- ๒ -

## รายชื่อผู้มีสิทธิ์เข้ารับการประเมินสมรรถนะ ครั้งที่ ๑
### ตำแหน่ง นักวิชาการสาธารณสุข
**แนบท้ายประกาศสำนักงานสาธารณสุขอำเภอดำเนินสะดวก ลงวันที่ ๑๑ พฤศจิกายน ๒๕๖๗**

| หมายเลขผู้สมัคร | ชื่อ-สกุล                   |
|---------------|--------------------------|
| ๐๑            | นายชัยธวัช กรุดโกศล        |
| ๐๒            | นางสาวจิรัชญา บุญเพ็ญ      |
| ๐๓            | นางสาวณัฐกานต์ ทินโน      |

**หมายเหตุ** รายชื่อทั้งหมดเรียงตามลำดับการสมัคร

---

## ตารางกำหนดการสรรหาและเลือกสรรลูกจ้างชั่วคราว (รายวัน) การประเมินสมรรถนะ ครั้งที่ ๑

### กำหนดการ
| วันและเวลา          | สถานที่                     | รายละเอียดกิจกรรม                                          | หมายเหตุ                  |
|--------------------|----------------------------|----------------------------------------------------------|-------------------------|
| ๑๔ พฤศจิกายน ๒๕๖๗ ๙.๐๐-๑๑.๐๐ น. | สำนักงานสาธารณสุขอำเภอดำเนินสะดวก | ประเมินสมรรถนะ ครั้งที่ ๑ สำหรับตำแหน่ง นักวิชาการสาธารณสุข | รายงานตัวก่อนเข้าห้องสอบ ๘.๓๐ น. |

### รายละเอียดการประเมิน
| หลักเกณฑ์การเลือกสรร                       | คะแนนเต็ม | วิธีการประเมิน |
|------------------------------------------|---------|---------------|
| การประเมินสมรรถนะ ครั้งที่ ๑               | ๑๐๐     | สอบข้อเขียน    |
| - สมรรถนะที่ ๑ ความรู้ความสามารถทั่วไป      | ๔๐      |               |
| - สมรรถนะที่ ๒ ความรู้ความสามารถเฉพาะตำแหน่ง | ๖๐      |               |

### หมายเหตุ
๑. ไม่อนุญาตให้ผู้สอบเข้าห้องสอบหลังจากเวลาที่กำหนดเริ่มสอบผ่านไปแล้ว ๑๕ นาที ยกเว้นคณะกรรมการดำเนินการสอบพิจารณาเห็นว่าเป็นเหตุสุดวิสัย และอนุมัติให้เข้าสอบได้
๒. ให้ผู้สอบนำบัตรประจำตัวประชาชนมาเพื่อใช้ยืนยันตัวตนในการเข้าสอบ
๓. ห้ามนำโทรศัพท์มือถือ อุปกรณ์การติดต่อสื่อสาร กล้องถ่ายภาพ นาฬิกาอิเล็กทรอนิกส์ทุกชนิดรวมทั้ง เอกสาร กระเป๋าถือ ย่าม เข้าห้องสอบ
๔. ในระหว่างการสอบ ถ้าต้องการสิ่งใดให้ยกมือขึ้น เพื่อให้กรรมการคุมสอบทราบ
๕. โปรดรักษาความสงบเมื่ออยู่ในห้องสอบ และขณะที่อยู่บริเวณสถานที่สอบ
๖. ห้ามมีการทุจริตไม่ว่ากรณีใด ๆ
๗. เมื่อหมดเวลาสอบ ให้หยุดทำข้อสอบทันที
๘. ห้ามนำข้อสอบหรือส่วนหนึ่งส่วนใดของข้อสอบออกจากห้องสอบโดยเด็ดขาด
๙. ในการเข้าสอบ ผู้เข้าสอบจะต้องแต่งกายสุภาพ ห้ามสวมเสื้อยืด หรือปล่อยชายเสื้อ สำหรับผู้หญิงให้ใส่กระโปรง หรือสวมกางเกงทรงสุภาพ มิฉะนั้นอาจไม่ได้รับอนุญาตให้เข้าห้องสอบ

---

## ตาราง กำหนดการสรรหาและเลือกสรรลูกจ้างชั่วคราวรายวัน
### แนบท้ายประกาศสำนักงานสาธารณสุขอำเภอดำเนินสะดวก
### เรื่อง รับสมัครบุคคลเพื่อสรรหาและเลือกสรรเป็นลูกจ้างชั่วคราว (รายวัน)

| วันและเวลา           | สถานที่                     | รายละเอียดกิจกรรม                                                                | หมายเหตุ |
|---------------------|----------------------------|---------------------------------------------------------------------------------|----------|
| ๓๐ ต.ค. ๖๗ - ๘ พ.ย. ๖๗ ๘.๓๐-๑๖.๐๐ น. | สำนักงานสาธารณสุขอำเภอดำเนินสะดวก | ขอรับและยื่นใบสมัคร                                                              |          |
| ๑๑ พ.ย. ๖๗ ๑๐.๐๐ น.  | สำนักงานสาธารณสุขอำเภอดำเนินสะดวก | ประกาศรายชื่อผู้มีสิทธิเข้ารับการประเมินความรู้ความสามารถทักษะ และสมรรถนะ           |          |
| ๑๔ พ.ย. ๖๗ ๙.๐๐-๑๒.๐๐ น. | สำนักงานสาธารณสุขอำเภอดำเนินสะดวก | ประเมินสมรรถนะ ครั้งที่ ๑ สำหรับตำแหน่ง นักวิชาการสาธารณสุข                       |          |
| ๑๔ พ.ย. ๖๗ ๑๓.๐๐-๑๖.๐๐ น. | สำนักงานสาธารณสุขอำเภอดำเนินสะดวก | การประเมินสมรรถนะ สำหรับตำแหน่ง พยาบาลวิชาชีพ (สอบสัมภาษณ์)                      |          |
| ๑๘ พ.ย. ๖๗ ๑๐.๐๐ น.  | สำนักงานสาธารณสุขอำเภอดำเนินสะดวก | ประกาศรายชื่อผู้มีสิทธิเข้ารับการประเมินความรู้ความสามารถทักษะ และสมรรถนะ ครั้งที่ ๒ สำหรับตำแหน่ง นักวิชาการสาธารณสุข |          |
| ๒๐ พ.ย. ๖๗ ๙.๐๐-๑๒.๐๐ น. | สำนักงานสาธารณสุขอำเภอดำเนินสะดวก | การประเมินสมรรถนะ ครั้งที่ ๒ สำหรับตำแหน่ง นักวิชาการสาธารณสุข (สอบสัมภาษณ์)      |          |
| ๒๒ พ.ย. ๖๗ ๙.๐๐ น.   | สำนักงานสาธารณสุขอำเภอดำเนินสะดวก | ประกาศรายชื่อผู้ผ่านการเลือกสรรตามลำดับ                                           |          |
| ๒๖ พ.ย. ๖๗ ๙.๐๐-๑๒.๐๐ น. | สำนักงานสาธารณสุขอำเภอดำเนินสะดวก | รายงานตัว และเลือกสถานที่ปฏิบัติงาน สำหรับลูกจ้างชั่วคราวรายวันทุกตำแหน่ง         |          |

### หมายเหตุ
๑. กำหนดการอาจเปลี่ยนแปลงได้ตามความเหมาะสม ซึ่งสำนักงานสาธารณสุขอำเภอดำเนินสะดวกจะประกาศให้ทราบต่อไป
๒. สามารถดูประกาศต่าง ๆ ได้ที่เว็บไซต์สำนักงานสาธารณสุขอำเภอดำเนินสะดวก (http://www.dndpho.org)

In [ ]:
class DocumentSummary(BaseModel):
    title: str
    summary: str
    key_points: list[str]
    action_items: list[str]


document_summary = gemini_client.beta.chat.completions.parse(
    model=GEMINI_MODEL,
    messages=summary_messages + [
        {
            "role": "user",
            "content": "สรุปเอกสารนี้ตาม schema ที่กำหนดและตอบเป็นภาษาไทย",
        }
    ],
    response_format=DocumentSummary,
)

pretty_print_json(document_summary.choices[0].message.parsed.model_dump())

{
  "title": "ประกาศรายชื่อผู้มีสิทธิเข้าประเมินสมรรถนะ ครั้งที่ ๑ ในการรับสมัครลูกจ้างชั่วคราว (รายวัน)",
  "summary": "เอกสารนี้ประกาศรายชื่อผู้มีสิทธิเข้ารับการประเมินสมรรถนะเพื่อคัดเลือกเป็นลูกจ้างชั่วคราว (รายวัน) ตำแหน่งพยาบาลวิชาชีพและนักวิชาการสาธารณสุข โดยระบุผู้สมัครที่มีสิทธิเข้ารับการประเมิน กำหนดการ และข้อปฏิบัติที่สำคัญในการสอบ",
  "key_points": [
    "ตำแหน่งที่เปิดรับคือ พยาบาลวิชาชีพ (2 อัตรา) และนักวิชาการสาธารณสุข (2 อัตรา) โดยจะรับเป็นลูกจ้างชั่วคราวรายวัน",
    "สำหรับตำแหน่งพยาบาลวิชาชีพ เอกสารระบุว่า 'ไม่มีผู้สมัคร' ที่มีสิทธิเข้ารับการประเมิน",
    "สำหรับตำแหน่งนักวิชาการสาธารณสุข มีผู้มีสิทธิเข้ารับการประเมินจำนวน 3 ราย ได้แก่ นายชัยธวัช กรุดโกศล, นางสาวจิรัชญา บุญเพ็ญ และนางสาวณัฐกานต์ ทินโน",
    "การประเมินสมรรถนะ ครั้งที่ 1 (สอบข้อเขียน) สำหรับตำแหน่งนักวิชาการสาธารณสุข จะจัดขึ้นในวันที่ 14 พฤศจิกายน 2567 เวลา 9.00-11.00 น. ณ สำนักงานสาธารณสุขอำเภอดำเนินสะดวก",
    "การประเมินสมรรถนะ (สอบสัมภาษณ์) สำหรับตำแหน่งพยาบาลวิชาชีพ จะจัดขึ้นในวันที่ 14 พฤศจิกายน 2

## 9) สรุป

จาก notebook นี้ เราได้เห็น workflow หลักของการทำงานกับ LLM:
- เริ่มจาก basic chat
- เพิ่ม context ด้วย history
- ควบคุมรูปแบบผลลัพธ์ด้วย JSON และ structured output
- ขยายความสามารถด้วย tool calling
- ใช้ multimodal model เพื่ออ่านเอกสารและทำ OCR

แนวคิดสำคัญคือเลือกเครื่องมือให้เหมาะกับงาน ไม่ใช่ใช้ prompt แบบเดียวกับทุกปัญหา
